### Merging The four tables together



In [5]:
# MERGE EXPLANATION:
# .merge() combines two DataFrames based on common columns (Entity, Year, Code)
# - When a match is found (same Entity+Year+Code in both tables): combines rows horizontally, 
#   keeping the common columns once and adding all feature columns from both tables
# - When no match: behavior depends on 'how' parameter
#
# how='inner': Only keeps rows that exist in BOTH tables (strictest, smallest dataset, minimal NaN)
# how='outer': Keeps ALL rows from both tables (largest dataset, adds NaN for missing data)
# how='left':  Keeps ALL rows from left (first) table, only matching rows from right (second) table
#
# Suffixes handle duplicate column names when merging.
# When both tables have a column with the same name (besides the merge keys like Entity, Year, Code), pandas adds suffixes to distinguish them.

import pandas as pd

basic_education = pd.read_csv("archive/1- share-of-the-world-population-with-at-least-basic-education.csv")
learning_years = pd.read_csv("archive/2- learning-adjusted-years-of-school-lays.csv")
out_school = pd.read_csv("archive/3- number-of-out-of-school-children.csv")
gap = pd.read_csv("archive/4- gender-gap-education-levels.csv")

combined = basic_education.merge(gap, on=['Entity', 'Year', 'Code'], how='outer', suffixes=('_basic', '_gap'))
combined = combined.merge(out_school, on=['Entity', 'Year', 'Code'], how='outer', suffixes=('', '_out'))
combined = combined.merge(learning_years, on=['Entity', 'Year', 'Code'], how='outer', suffixes=('', '_learn'))


display(combined.shape)
print("missing values per column:")
print(combined.isnull().sum())
display(combined.head())
display(combined.tail())
display(combined.columns)

(9945, 14)

missing values per column:
Entity                                                                                                   0
Code                                                                                                  1092
Year                                                                                                     0
Share of population with no formal education, 1820-2020                                               9477
Share of population with some formal education, 1820-2020                                             9477
Combined gross enrolment ratio for tertiary education, female                                         1694
Combined gross enrolment ratio for tertiary education, male                                           1694
Combined total net enrolment rate, secondary, male                                                    3407
Combined total net enrolment rate, secondary, female                                                  3407
Combined t

,Entity,Code,Year,"Share of population with no formal education, 1820-2020","Share of population with some formal education, 1820-2020","Combined gross enrolment ratio for tertiary education, female","Combined gross enrolment ratio for tertiary education, male","Combined total net enrolment rate, secondary, male","Combined total net enrolment rate, secondary, female","Combined total net enrolment rate, primary, female","Combined total net enrolment rate, primary, male","Out-of-school children, adolescents and youth of primary and secondary school age, male (number)","Out-of-school children, adolescents and youth of primary and secondary school age, female (number)",Learning-Adjusted Years of School
0,Afghanistan,AFG,1820,NaN,NaN,0.0,0.01,0.01,0.0,0.0,0.01,NaN,NaN,NaN
1,Afghanistan,AFG,1825,NaN,NaN,0.0,0.01,0.01,0.0,0.0,0.01,NaN,NaN,NaN
2,Afghanistan,AFG,1830,NaN,NaN,0.0,0.01,0.01,0.0,0.0,0.01,NaN,NaN,NaN
3,Afghanistan,AFG,1835,NaN,NaN,0.0,0.01,0.01,0.0,0.0,0.01,NaN,NaN,NaN
4,Afghanistan,AFG,1840,NaN,NaN,0.0,0.01,0.01,0.0,0.0,0.01,NaN,NaN,NaN


,Entity,Code,Year,"Share of population with no formal education, 1820-2020","Share of population with some formal education, 1820-2020","Combined gross enrolment ratio for tertiary education, female","Combined gross enrolment ratio for tertiary education, male","Combined total net enrolment rate, secondary, male","Combined total net enrolment rate, secondary, female","Combined total net enrolment rate, primary, female","Combined total net enrolment rate, primary, male","Out-of-school children, adolescents and youth of primary and secondary school age, male (number)","Out-of-school children, adolescents and youth of primary and secondary school age, female (number)",Learning-Adjusted Years of School
9940,Zimbabwe,ZWE,2015,4.0,96.0,9.17213,10.91966,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9941,Zimbabwe,ZWE,2016,NaN,NaN,8.54473,7.76729,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9942,Zimbabwe,ZWE,2017,NaN,NaN,9.68927,8.00596,NaN,NaN,NaN,NaN,NaN,NaN,6.35000
9943,Zimbabwe,ZWE,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.00661
9944,Zimbabwe,ZWE,2020,3.0,97.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.00661


Index(['Entity', 'Code', 'Year',
       'Share of population with no formal education, 1820-2020',
       'Share of population with some formal education, 1820-2020',
       'Combined gross enrolment ratio for tertiary education, female',
       'Combined gross enrolment ratio for tertiary education, male',
       'Combined total net enrolment rate, secondary, male',
       'Combined total net enrolment rate, secondary, female',
       'Combined total net enrolment rate, primary, female',
       'Combined total net enrolment rate, primary, male',
       'Out-of-school children, adolescents and youth of primary and secondary school age, male (number)',
       'Out-of-school children, adolescents and youth of primary and secondary school age, female (number)',
       'Learning-Adjusted Years of School'],
      dtype='object')

### Computing extra features to match assignment requirements

In [6]:
# positive = more males, negative = more females
combined['tertiary_gender_gap'] = combined['Combined gross enrolment ratio for tertiary education, male'] - combined['Combined gross enrolment ratio for tertiary education, female']
combined['secondary_gender_gap'] = combined['Combined total net enrolment rate, secondary, male'] - combined['Combined total net enrolment rate, secondary, female']
combined['primary_gender_gap'] = combined['Combined total net enrolment rate, primary, male'] - combined['Combined total net enrolment rate, primary, female']

# total out of school
combined['total_out_of_school'] = combined['Out-of-school children, adolescents and youth of primary and secondary school age, male (number)'] + combined['Out-of-school children, adolescents and youth of primary and secondary school age, female (number)']

# Total tertiary enrollment
combined['total_tertiary_enrollment'] = combined['Combined gross enrolment ratio for tertiary education, male'] + combined['Combined gross enrolment ratio for tertiary education, female']

# Decade
combined['decade'] = (combined['Year'] // 10) * 10

# Check the updated dataset
display(combined.shape)



(9945, 20)

### Creating the final table

In [7]:
combined.to_csv('combined_dataset.csv', index=False)